# Policy Network for Collision Avoidance via SRP Control

Train a neural network policy to maximize deviation from baseline orbit at collision time.

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax
import pickle
from pathlib import Path
import time as time_module

import base, designer, nn, physics, integrator, force_modeling
from utils import load_trajectory, sample_moonsun_cubic

jax.config.update("jax_enable_x64", True)
plt.rcParams['figure.figsize'] = (10, 6)

## Configuration

In [ ]:
# Simulation
SIM_HOURS = 32
COLLIDE_HOUR = 16
RECOVER_HOUR = 17
STEP_SIZE = 120.0
CONTROL_DS = 30

# Loss weights
RECOVER_WEIGHT = 2.0
TERMINAL_WEIGHT = 5.0
COLLISION_WEIGHT = 17.0

# Training
POLICY_EPOCHS = 50000
POLICY_LR = 3e-4
GRAD_CLIP = 1.0
NUM_SEGMENTS = 10

# Sequence optimization
SEQ_ITERS = 1000
SEQ_LR = 1e-2

# Weights
WEIGHTS_PATH = Path(base.drive_path + "nn_weights/policy_collision_avoidance.pkl")
LOAD_PRETRAINED = True    # Skip training, use saved weights
RESUME_TRAINING = False

print(f"Config: {SIM_HOURS}h, collision@{COLLIDE_HOUR}h, {POLICY_EPOCHS} epochs")

## Setup

In [ ]:
# Object setup
OBJECT_NAME = 'cuboid_6faces/cuboid_6faces'
DESIGN_NAME = 'face_albedo_sweep'
face_specs = [designer.make_face_albedo_spec(f'Face_{i+1}') for i in range(6)]
cuboid_design = designer.build_face_albedo_design(OBJECT_NAME, face_specs)
FACE_COUNT = int(cuboid_design.shape)
FACE_LIMITS = jnp.ones(FACE_COUNT, dtype=jnp.float64)

# Load NN proxy
nn_dir = Path(base.get_nn_weight_path(OBJECT_NAME, DESIGN_NAME))
combined_bundle = {
    "params": tuple(jnp.asarray(p, dtype=jnp.float64) for p in nn.load_nn_parameters(nn_dir / "combined_parameters.bin")),
    "feature_stats": tuple(jnp.asarray(s, dtype=jnp.float64) for s in nn.load_nn_parameters(nn_dir / "combined_ray_stats.bin")),
    "target_stats": tuple(jnp.asarray(s, dtype=jnp.float64) for s in nn.load_nn_parameters(nn_dir / "combined_target_stats.bin"))
}
print(f"NN proxy: {FACE_COUNT} faces")

In [ ]:
# Simulation params
STEP_COUNT = int(SIM_HOURS * 3600 / STEP_SIZE)
CONTROL_COUNT_DS = (STEP_COUNT) // CONTROL_DS + 1
COLLIDE_TIME_S = COLLIDE_HOUR * 3600.0
RECOVER_START_S = RECOVER_HOUR * 3600.0
COLLIDE_WINDOW_S = 3600.0

# Spacecraft
SC_MASS = 100.0
MOMENT_OF_INERTIA = jnp.array([1.0, 1.0, 1.0])
TORQUE_GAIN = 1.0
BOX_AREA = 50.0

# Trajectory
trajectory = load_trajectory(base.drive_path + "trajectories/gps08.csv", limit=None)
sun_times = jnp.asarray(trajectory.time, dtype=jnp.float64)
sun_positions = jnp.asarray(trajectory.sun, dtype=jnp.float64)

print(f"Simulation: {STEP_COUNT} steps, {CONTROL_COUNT_DS} control points")

## Core Functions

In [ ]:
def eval_proxy(bundle, sun_body, albedo):
    features = jnp.concatenate([sun_body, 0.5 * (albedo / FACE_LIMITS + 1.0)])
    pred = nn.mlp(nn.whiten_with_stats(features, bundle["feature_stats"]), bundle["params"])
    pred_real = nn.unwhiten(0.5 * (pred + 1.0), bundle["target_stats"])
    return BOX_AREA * pred_real[:3], TORQUE_GAIN * pred_real[3:]

def softsign(x): return x / (1.0 + jnp.abs(x))
def params_to_albedo(raw): return jnp.clip(softsign(jnp.asarray(raw, dtype=jnp.float64)), 0.0, FACE_LIMITS)

def solve_with_history(z0, t_start, t_stop, ode, ode_param, step_count, *args):
    step_size = (t_stop - t_start) / step_count
    def step(carry, time):
        new_state = integrator.integrate_rk4(carry, time, ode, ode_param, step_size, *args)
        return new_state, carry
    times = t_start + step_size * jnp.arange(step_count)
    final_state, history = jax.lax.scan(step, z0, times)
    history_full = jax.tree.map(lambda h, f: jnp.concatenate([h, f[jnp.newaxis, ...]], axis=0), history, final_state)
    return final_state, history_full

In [ ]:
def build_srp_provider(albedo_table, control_times):
    def provider(position, time, euler_angles):
        idx = jnp.clip(jnp.searchsorted(control_times, time, side="right"), 1, len(control_times) - 1)
        t_left, t_right = control_times[idx - 1], control_times[idx]
        alpha = jnp.clip((time - t_left) / (t_right - t_left + 1e-9), 0.0, 1.0)
        albedo = (1 - alpha) * albedo_table[idx - 1] + alpha * albedo_table[idx]
        
        sun_pos = sample_moonsun_cubic(time, sun_positions, sun_times)
        sun_dir = (sun_pos - position) / jnp.maximum(jnp.linalg.norm(sun_pos - position), 1e-12)
        R = physics.rotation_matrix_from_euler(euler_angles)
        force_body, torque_body = eval_proxy(combined_bundle, R.T @ sun_dir, albedo)
        return R @ force_body / SC_MASS, torque_body
    return provider

In [ ]:
def make_segment_simulator(seg_idx):
    pos0 = jnp.asarray(trajectory.position[seg_idx], dtype=jnp.float64)
    vel0 = jnp.asarray(trajectory.velocity[seg_idx], dtype=jnp.float64)
    t0 = float(trajectory.time[seg_idx])
    
    # Initial attitude
    sun_seg = jnp.asarray(trajectory.sun[seg_idx], dtype=jnp.float64)
    z = -pos0 / jnp.linalg.norm(pos0)
    y = jnp.cross(sun_seg / jnp.linalg.norm(sun_seg), -z)
    y = y / jnp.linalg.norm(y)
    x = jnp.cross(y, z)
    R = jnp.column_stack([x, y, z])
    angles0 = jnp.array([jnp.arctan2(R[2,1], R[2,2]), jnp.arcsin(jnp.clip(-R[2,0], -1, 1)), jnp.arctan2(R[1,0], R[0,0])])
    omega0 = jnp.array([jnp.deg2rad(0.05), jnp.deg2rad(-0.03), jnp.deg2rad(0.02)])
    
    state0 = (pos0, vel0, angles0, omega0)
    sim_times = t0 + jnp.arange(STEP_COUNT + 1) * STEP_SIZE
    control_times = t0 + jnp.arange(CONTROL_COUNT_DS) * STEP_SIZE * CONTROL_DS
    t_stop = t0 + STEP_COUNT * STEP_SIZE
    
    def simulate(raw_table):
        albedo = params_to_albedo(raw_table)
        provider = build_srp_provider(albedo, control_times)
        _, hist = solve_with_history(state0, t0, t_stop, force_modeling.two_body_srp_with_torque, 
                                      provider, STEP_COUNT, MOMENT_OF_INERTIA)
        return _, hist
    
    # Baseline
    _, baseline_hist = simulate(jnp.zeros((CONTROL_COUNT_DS, FACE_COUNT)))
    baseline_pos = baseline_hist[0]
    
    return simulate, baseline_pos, sim_times, t0, state0, control_times

In [ ]:
def make_loss_fn(simulate, baseline_pos, sim_times, t0):
    coll_weight = jnp.exp(-0.5 * ((sim_times - (t0 + COLLIDE_TIME_S)) / COLLIDE_WINDOW_S) ** 2)
    rec_mask = (sim_times >= (t0 + RECOVER_START_S)).astype(jnp.float64)
    
    def loss_fn(raw_table):
        _, hist = simulate(raw_table)
        pos = hist[0]
        dist = jnp.sqrt(jnp.sum((pos - baseline_pos) ** 2, axis=1) + 1e-12) * 1e-3
        rec_err = jnp.sum(dist * rec_mask) / jnp.maximum(jnp.sum(rec_mask), 1.0)
        term_err = jnp.sqrt(jnp.sum((pos[-1] - baseline_pos[-1]) ** 2) + 1e-12) * 1e-3
        coll_rew = jnp.sum(dist * coll_weight) / jnp.sum(coll_weight)
        return RECOVER_WEIGHT * rec_err + TERMINAL_WEIGHT * term_err - COLLISION_WEIGHT * coll_rew, (rec_err, term_err, coll_rew)
    return loss_fn

## Policy Network

In [ ]:
def init_policy(key):
    k1, k2, k3 = jax.random.split(key, 3)
    return {
        'W1': jax.random.normal(k1, (12, 64)) * jnp.sqrt(2.0 / 76), 'b1': jnp.zeros(64),
        'W2': jax.random.normal(k2, (64, 64)) * jnp.sqrt(2.0 / 128), 'b2': jnp.zeros(64),
        'W3': jax.random.normal(k3, (64, FACE_COUNT)) * jnp.sqrt(2.0 / 70), 'b3': jnp.zeros(FACE_COUNT)
    }

def policy_forward(params, features):
    h = jax.nn.relu(features @ params['W1'] + params['b1'])
    h = jax.nn.relu(h @ params['W2'] + params['b2'])
    return jax.nn.sigmoid(h @ params['W3'] + params['b3'])

def extract_features(position, euler_angles, omega, time, baseline_pos, t0):
    R = physics.rotation_matrix_from_euler(euler_angles)
    sun_body = R.T @ jnp.array([0.0, -1.0, 0.0])
    d_base = jnp.sqrt(jnp.sum((position - baseline_pos) ** 2) + 1e-12) * 1e-3
    phase = (time - t0) / (SIM_HOURS * 3600.0)
    prox = jnp.exp(-0.5 * ((time - t0 - COLLIDE_TIME_S) / COLLIDE_WINDOW_S) ** 2)
    return jnp.concatenate([sun_body, euler_angles, omega, jnp.array([d_base, phase, prox])])

print(f"Policy: {sum(p.size for p in init_policy(jax.random.PRNGKey(0)).values())} params")

In [ ]:
def make_policy_loss_fn(seg_idx):
    simulate, baseline_pos, sim_times, t0, state0, control_times = make_segment_simulator(seg_idx)
    pos0, vel0, angles0, omega0 = state0
    t_stop = t0 + STEP_COUNT * STEP_SIZE
    coll_weight = jnp.exp(-0.5 * ((sim_times - (t0 + COLLIDE_TIME_S)) / COLLIDE_WINDOW_S) ** 2)
    rec_mask = (sim_times >= (t0 + RECOVER_START_S)).astype(jnp.float64)
    
    def build_policy_provider(policy_params):
        def provider(position, time, euler_angles):
            idx = jnp.clip(jnp.searchsorted(sim_times, time, side="right") - 1, 0, STEP_COUNT)
            features = extract_features(position, euler_angles, omega0, time, baseline_pos[idx], t0)
            albedo = policy_forward(policy_params, features)
            
            sun_pos = sample_moonsun_cubic(time, sun_positions, sun_times)
            sun_dir = (sun_pos - position) / jnp.maximum(jnp.linalg.norm(sun_pos - position), 1e-12)
            R = physics.rotation_matrix_from_euler(euler_angles)
            force_body, torque_body = eval_proxy(combined_bundle, R.T @ sun_dir, albedo)
            return R @ force_body / SC_MASS, torque_body
        return provider
    
    def loss_fn(policy_params):
        provider = build_policy_provider(policy_params)
        _, hist = solve_with_history(state0, t0, t_stop, force_modeling.two_body_srp_with_torque,
                                      provider, STEP_COUNT, MOMENT_OF_INERTIA)
        pos = hist[0]
        dist = jnp.sqrt(jnp.sum((pos - baseline_pos) ** 2, axis=1) + 1e-12) * 1e-3
        rec_err = jnp.sum(dist * rec_mask) / jnp.maximum(jnp.sum(rec_mask), 1.0)
        term_err = jnp.sqrt(jnp.sum((pos[-1] - baseline_pos[-1]) ** 2) + 1e-12) * 1e-3
        coll_rew = jnp.sum(dist * coll_weight) / jnp.sum(coll_weight)
        return RECOVER_WEIGHT * rec_err + TERMINAL_WEIGHT * term_err - COLLISION_WEIGHT * coll_rew, (rec_err, term_err, coll_rew)
    
    return loss_fn, baseline_pos, sim_times, t0

## Training

In [ ]:
loaded_params = None
if (LOAD_PRETRAINED or RESUME_TRAINING) and WEIGHTS_PATH.exists():
    with open(WEIGHTS_PATH, 'rb') as f: ckpt = pickle.load(f)
    loaded_params = ckpt['params']
    print(f"Loaded: epoch {ckpt.get('epoch', '?')}, loss {ckpt.get('best_loss', '?'):.4f}")
    if LOAD_PRETRAINED: print("Skipping training")
elif LOAD_PRETRAINED:
    print(f"No weights at {WEIGHTS_PATH}")
    LOAD_PRETRAINED = False

In [ ]:
if not LOAD_PRETRAINED:
    max_start = len(trajectory.time) - STEP_COUNT - 100
    train_idx = np.linspace(0, max(1, max_start), NUM_SEGMENTS, dtype=int)
    print(f"Training on {NUM_SEGMENTS} segments: {train_idx.tolist()}")
    
    print("\nCompiling...")
    compiled = []
    t_start = time_module.time()
    for i, seg in enumerate(train_idx):
        loss_fn, _, _, _ = make_policy_loss_fn(seg)
        lg = jax.jit(jax.value_and_grad(loss_fn, has_aux=True))
        _ = lg(init_policy(jax.random.PRNGKey(0)))
        compiled.append(lg)
        print(f"  {i+1}/{NUM_SEGMENTS} (idx={seg}) [{time_module.time()-t_start:.1f}s]")
    print(f"Done: {time_module.time()-t_start:.1f}s")

In [ ]:
if not LOAD_PRETRAINED:
    params = loaded_params if RESUME_TRAINING and loaded_params else init_policy(jax.random.PRNGKey(42))
    opt = optax.chain(optax.clip_by_global_norm(GRAD_CLIP), optax.adam(POLICY_LR))
    opt_state = opt.init(params)
    
    loss_trace, grad_trace, seg_trace = [], [], []
    best_params, best_loss = params, float('inf')
    seg_best = {i: float('inf') for i in range(NUM_SEGMENTS)}
    
    print(f"\nTraining: {POLICY_EPOCHS} epochs")
    t_train = time_module.time()
    
    for epoch in range(POLICY_EPOCHS):
        seg_i = epoch % NUM_SEGMENTS
        (loss, _), grads = compiled[seg_i](params)
        updates, opt_state = opt.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        
        grad_norm = min(jnp.sqrt(sum(jnp.sum(g**2) for g in jax.tree_util.tree_leaves(grads))), GRAD_CLIP)
        loss_trace.append(float(loss)); grad_trace.append(float(grad_norm)); seg_trace.append(seg_i)
        
        if loss < seg_best[seg_i]: seg_best[seg_i] = float(loss)
        avg = np.mean(list(seg_best.values()))
        if avg < best_loss: best_loss, best_params = avg, jax.tree_util.tree_map(lambda x: x.copy(), params)
        
        if epoch % 5000 == 0 or epoch == POLICY_EPOCHS - 1:
            print(f"epoch={epoch:05d} loss={loss:.4f} avg={avg:.4f} [{time_module.time()-t_train:.1f}s]")
    
    print(f"\nDone: {time_module.time()-t_train:.1f}s, best={best_loss:.4f}")
else:
    best_params, best_loss = loaded_params, ckpt.get('best_loss', 0)
    loss_trace, grad_trace, seg_trace = [], [], []

In [ ]:
WEIGHTS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_PATH, 'wb') as f:
    pickle.dump({'params': best_params, 'epoch': POLICY_EPOCHS if not LOAD_PRETRAINED else ckpt.get('epoch', 0),
                 'best_loss': best_loss}, f)
print(f"Saved to {WEIGHTS_PATH}")

In [ ]:
if loss_trace:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].scatter(range(len(loss_trace)), loss_trace, c=plt.cm.tab10(np.array(seg_trace)/max(max(seg_trace),1)), s=1, alpha=0.5)
    ax[0].axhline(best_loss, color='r', ls='--', label=f'Best: {best_loss:.4f}')
    ax[0].set(xlabel='Epoch', ylabel='Loss', title='Training Loss'); ax[0].legend(); ax[0].grid(True, alpha=0.3)
    ax[1].plot(grad_trace, alpha=0.7); ax[1].set(xlabel='Epoch', ylabel='Grad', title='Gradient'); ax[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

## Evaluation

In [ ]:
print("Evaluating...")
loss_fn, baseline_pos, sim_times, t0 = make_policy_loss_fn(0)
simulate, _, _, _, state0, control_times = make_segment_simulator(0)
t_stop = t0 + STEP_COUNT * STEP_SIZE

# Policy
def pol_provider(policy_params):
    omega0 = state0[3]
    def provider(position, time, euler_angles):
        idx = jnp.clip(jnp.searchsorted(sim_times, time, side="right") - 1, 0, STEP_COUNT)
        features = extract_features(position, euler_angles, omega0, time, baseline_pos[idx], t0)
        albedo = policy_forward(policy_params, features)
        sun_pos = sample_moonsun_cubic(time, sun_positions, sun_times)
        sun_dir = (sun_pos - position) / jnp.maximum(jnp.linalg.norm(sun_pos - position), 1e-12)
        R = physics.rotation_matrix_from_euler(euler_angles)
        force_body, torque_body = eval_proxy(combined_bundle, R.T @ sun_dir, albedo)
        return R @ force_body / SC_MASS, torque_body
    return provider

_, pol_hist = solve_with_history(state0, t0, t_stop, force_modeling.two_body_srp_with_torque,
                                  pol_provider(best_params), STEP_COUNT, MOMENT_OF_INERTIA)
pol_dev = np.sqrt(np.sum((np.asarray(pol_hist[0]) - np.asarray(baseline_pos)) ** 2, axis=1))

# Sequence
print("Sequence opt...")
loss_fn_seq = make_loss_fn(simulate, baseline_pos, sim_times, t0)
lg_seq = jax.jit(jax.value_and_grad(loss_fn_seq, has_aux=True))
seq_p = jax.random.uniform(jax.random.PRNGKey(0), (CONTROL_COUNT_DS, FACE_COUNT), minval=0.5, maxval=2.0)
seq_opt = optax.adam(SEQ_LR)
seq_st = seq_opt.init(seq_p)
for _ in range(SEQ_ITERS):
    (_, _), g = lg_seq(seq_p)
    u, seq_st = seq_opt.update(g, seq_st, seq_p)
    seq_p = optax.apply_updates(seq_p, u)
_, seq_hist = simulate(seq_p)
seq_dev = np.sqrt(np.sum((np.asarray(seq_hist[0]) - np.asarray(baseline_pos)) ** 2, axis=1))

# Random
rand_p = jax.random.uniform(jax.random.PRNGKey(123), (CONTROL_COUNT_DS, FACE_COUNT), minval=0.5, maxval=2.0)
_, rand_hist = simulate(rand_p)
rand_dev = np.sqrt(np.sum((np.asarray(rand_hist[0]) - np.asarray(baseline_pos)) ** 2, axis=1))

coll_idx = int(COLLIDE_TIME_S / STEP_SIZE)
print(f"\nResults:")
print(f"  Policy:   peak={pol_dev.max():.1f}m, @coll={pol_dev[coll_idx]:.1f}m")
print(f"  Sequence: peak={seq_dev.max():.1f}m, @coll={seq_dev[coll_idx]:.1f}m")
print(f"  Random:   peak={rand_dev.max():.1f}m, @coll={rand_dev[coll_idx]:.1f}m")

In [ ]:
time_h = (np.asarray(sim_times) - t0) / 3600
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(time_h, pol_dev, 'b-', lw=2, label=f'Policy ({pol_dev.max():.0f}m)')
ax.plot(time_h, seq_dev, 'g-', lw=2, label=f'Sequence ({seq_dev.max():.0f}m)')
ax.plot(time_h, rand_dev, 'gray', lw=1, alpha=0.7, label=f'Random ({rand_dev.max():.0f}m)')
ax.axvline(COLLIDE_HOUR, color='r', ls='--', alpha=0.5)
ax.set(xlabel='Hours', ylabel='Deviation (m)', title='Collision Avoidance')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## Generalization

In [ ]:
print("Generalization test...\n")
max_start = len(trajectory.time) - STEP_COUNT - 100
test_idx = np.linspace(0, max(1, max_start), 10, dtype=int)

results = []
for i, seg in enumerate(test_idx):
    loss_fn, baseline, sim_t, t0 = make_policy_loss_fn(seg)
    simulate, _, _, _, st0, ctrl_t = make_segment_simulator(seg)
    t_stop = t0 + STEP_COUNT * STEP_SIZE
    
    # Policy
    def mk_prov(params, omega0):
        def provider(pos, time, angles):
            idx = jnp.clip(jnp.searchsorted(sim_t, time, side="right") - 1, 0, STEP_COUNT)
            feat = extract_features(pos, angles, omega0, time, baseline[idx], t0)
            alb = policy_forward(params, feat)
            sun = sample_moonsun_cubic(time, sun_positions, sun_times)
            sun_d = (sun - pos) / jnp.maximum(jnp.linalg.norm(sun - pos), 1e-12)
            R = physics.rotation_matrix_from_euler(angles)
            f, torq = eval_proxy(combined_bundle, R.T @ sun_d, alb)
            return R @ f / SC_MASS, torq
        return provider
    
    _, ph = solve_with_history(st0, t0, t_stop, force_modeling.two_body_srp_with_torque,
                                mk_prov(best_params, st0[3]), STEP_COUNT, MOMENT_OF_INERTIA)
    pd = np.sqrt(np.sum((np.asarray(ph[0]) - np.asarray(baseline)) ** 2, axis=1))
    
    # Seq
    lf = make_loss_fn(simulate, baseline, sim_t, t0)
    lg = jax.jit(jax.value_and_grad(lf, has_aux=True))
    sp = jax.random.uniform(jax.random.PRNGKey(seg), (CONTROL_COUNT_DS, FACE_COUNT), minval=0.5, maxval=2.0)
    so = optax.adam(SEQ_LR); ss = so.init(sp)
    for _ in range(200):
        (_,_), g = lg(sp); u, ss = so.update(g, ss, sp); sp = optax.apply_updates(sp, u)
    _, sh = simulate(sp)
    sd = np.sqrt(np.sum((np.asarray(sh[0]) - np.asarray(baseline)) ** 2, axis=1))
    
    # Rand
    rp = jax.random.uniform(jax.random.PRNGKey(seg+100), (CONTROL_COUNT_DS, FACE_COUNT), minval=0.5, maxval=2.0)
    _, rh = simulate(rp)
    rd = np.sqrt(np.sum((np.asarray(rh[0]) - np.asarray(baseline)) ** 2, axis=1))
    
    ci = int(COLLIDE_TIME_S / STEP_SIZE)
    results.append({'pol': pd.max(), 'seq': sd.max(), 'rand': rd.max(), 'pol_c': pd[ci], 'seq_c': sd[ci]})


In [ ]:
print("\n" + "="*50)
print("SUMMARY")
print("="*50)
ap, as_, ar = np.mean([r['pol'] for r in results]), np.mean([r['seq'] for r in results]), np.mean([r['rand'] for r in results])
apc, asc = np.mean([r['pol_c'] for r in results]), np.mean([r['seq_c'] for r in results])
print(f"\nAvg peak: Policy={ap:.1f}m, Sequence={as_:.1f}m, Random={ar:.1f}m")
print(f"Avg @coll: Policy={apc:.1f}m, Sequence={asc:.1f}m")
print(f"\nPolicy vs Seq: {ap/as_*100:.1f}%")
print(f"Policy vs Random: {ar/ap:.1f}x better")

In [ ]:
# 10 subplots: Policy vs Sequence vs Random for each segment
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, seg in enumerate(test_idx):
    ax = axes[i]
    
    # Recompute trajectories for this segment
    loss_fn, baseline, sim_t, t0 = make_policy_loss_fn(seg)
    simulate, _, _, _, st0, _ = make_segment_simulator(seg)
    t_stop = t0 + STEP_COUNT * STEP_SIZE
    time_h = (np.asarray(sim_t) - t0) / 3600
    
    # Policy
    def mk_prov(params, omega0):
        def provider(pos, time, angles):
            idx = jnp.clip(jnp.searchsorted(sim_t, time, side="right") - 1, 0, STEP_COUNT)
            feat = extract_features(pos, angles, omega0, time, baseline[idx], t0)
            alb = policy_forward(params, feat)
            sun = sample_moonsun_cubic(time, sun_positions, sun_times)
            sun_d = (sun - pos) / jnp.maximum(jnp.linalg.norm(sun - pos), 1e-12)
            R = physics.rotation_matrix_from_euler(angles)
            f, torq = eval_proxy(combined_bundle, R.T @ sun_d, alb)
            return R @ f / SC_MASS, torq
        return provider
    
    _, ph = solve_with_history(st0, t0, t_stop, force_modeling.two_body_srp_with_torque,
                                mk_prov(best_params, st0[3]), STEP_COUNT, MOMENT_OF_INERTIA)
    pd = np.sqrt(np.sum((np.asarray(ph[0]) - np.asarray(baseline)) ** 2, axis=1))
    
    # Sequence
    lf = make_loss_fn(simulate, baseline, sim_t, t0)
    lg = jax.jit(jax.value_and_grad(lf, has_aux=True))
    sp = jax.random.uniform(jax.random.PRNGKey(seg), (CONTROL_COUNT_DS, FACE_COUNT), minval=0.5, maxval=2.0)
    so = optax.adam(SEQ_LR); ss = so.init(sp)
    for _ in range(200):
        (_,_), g = lg(sp); u, ss = so.update(g, ss, sp); sp = optax.apply_updates(sp, u)
    _, sh = simulate(sp)
    sd = np.sqrt(np.sum((np.asarray(sh[0]) - np.asarray(baseline)) ** 2, axis=1))
    
    # Random
    rp = jax.random.uniform(jax.random.PRNGKey(seg+100), (CONTROL_COUNT_DS, FACE_COUNT), minval=0.5, maxval=2.0)
    _, rh = simulate(rp)
    rd = np.sqrt(np.sum((np.asarray(rh[0]) - np.asarray(baseline)) ** 2, axis=1))
    
    # Plot
    ax.plot(time_h, pd, 'b-', lw=1.5, label=f'Policy ({pd.max():.0f}m)')
    ax.plot(time_h, sd, 'g-', lw=1.5, label=f'Seq ({sd.max():.0f}m)')
    ax.plot(time_h, rd, 'gray', lw=1, alpha=0.7, label=f'Rand ({rd.max():.0f}m)')
    ax.axvline(COLLIDE_HOUR, color='r', ls='--', alpha=0.5, lw=1)
    ax.set_title(f'Segment {i} (idx={seg})', fontsize=10)
    ax.set_xlabel('Hours', fontsize=8)
    ax.set_ylabel('Deviation (m)', fontsize=8)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=7)

plt.suptitle('Collision Avoidance: Policy vs Sequence vs Random (10 segments)', fontsize=14)
plt.tight_layout()
plt.savefig(str(base.PROJECT_ROOT / 'plots/policy_vs_seq_10segments.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved to plots/policy_vs_seq_10segments.png")